# Consolidated SDC Sweep Notebook -- All 5 Keys

Every sweep (Mock and real-channel) now loops over all 5 collected key pairs, not just key0. No finite_key data collected here -- placeholder and asymptotic only.

**Cost note**: real-channel sweeps default to `n_runs=5` per key/probability (not 20) to keep total FABRIC trial count manageable across 5 keys. Raise `REALCHANNEL_N_RUNS` below if you have time/budget to spare.

In [1]:
import sys, json, re, time, os
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession, SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.sweep_utils import (
    run_condition_sweep, summarize_outcomes,
    run_real_channel_trial, run_real_channel_reconciliation_trial,
    collect_key_pairs,
)

SLICE_NAME = 'qfabric-bb84-2'
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

probs_th = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
probs_recon = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]

MOCK_N_RUNS = 50          # cheap, local -- fine at full density per key
REALCHANNEL_N_RUNS = 100    # real FABRIC trials -- kept low since this now multiplies by 5 keys

print("Setup complete: slice, nodes, imports ready")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,a2959fe6-7d36-4264-b81f-0449a1506af7
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-09-23 02:53:18 +0000
Lease Start (UTC),2026-09-09 02:53:18 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


Setup complete: slice, nodes, imports ready


In [7]:
# --- Load existing key pairs, do NOT re-collect ---
metadata_path = PROJECT_DIR / "results" / "key_pairs_metadata.csv"

if not metadata_path.exists():
    raise FileNotFoundError(
        f"{metadata_path} not found -- collect_key_pairs was never run, or "
        f"results were deleted. Do NOT call collect_key_pairs() to fix this "
        f"unless you're certain you want 5 brand-new keys (which would silently "
        f"invalidate any existing sweep data still labeled key_index 0-4)."
    )

key_pairs_df = pd.read_csv(str(metadata_path))
print(key_pairs_df)

   index  m_sifted  k_pe  n_bits      qber  \
0      0      3894   390    3504  0.012821   
1      1      3788   379    3409  0.015831   
2      2      3734   374    3360  0.002674   
3      3      3813   382    3431  0.002618   
4      4      3841   385    3456  0.005195   

                                          alice_path  \
0  /home/fabric/work/qkd-dependability/results/al...   
1  /home/fabric/work/qkd-dependability/results/al...   
2  /home/fabric/work/qkd-dependability/results/al...   
3  /home/fabric/work/qkd-dependability/results/al...   
4  /home/fabric/work/qkd-dependability/results/al...   

                                            bob_path  \
0  /home/fabric/work/qkd-dependability/results/bo...   
1  /home/fabric/work/qkd-dependability/results/bo...   
2  /home/fabric/work/qkd-dependability/results/bo...   
3  /home/fabric/work/qkd-dependability/results/bo...   
4  /home/fabric/work/qkd-dependability/results/bo...   

                                      alice_raw_p

In [8]:
stdout, _ = bob.execute("ps aux | grep bob_cascade_driver", quiet=True)
print(stdout)

stdout, _ = bob.execute("ss -ltnp | grep 5200", quiet=True)
print(stdout if stdout.strip() else "Port 5200: clear")

ubuntu     37179  0.0  0.0   7764  3360 ?        Ss   13:35   0:00 bash -c ps aux | grep bob_cascade_driver
ubuntu     37181  0.0  0.0   7012  2076 ?        S    13:35   0:00 grep bob_cascade_driver

Port 5200: clear


In [9]:
alice.upload_file(str(PROJECT_DIR / "scripts" / "bob_cascade_driver.py"), "qfabric/scripts/bob_cascade_driver.py")
bob.upload_file(str(PROJECT_DIR / "scripts" / "bob_cascade_driver.py"), "qfabric/scripts/bob_cascade_driver.py")

<SFTPAttributes: [ size=13510 uid=1000 gid=1000 mode=0o100664 atime=1789132871 mtime=1789133719 ]>

In [10]:
from qne.cascade.parameter_estimation import split_pe_and_generation
import json as _json

RAW_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"
RAW_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"
GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

alice_key_full, alice_indices_full = key_from_sifted_json(str(RAW_ALICE), "alice_bits")
bob_key_full, bob_indices_full = key_from_sifted_json(str(RAW_BOB), "bob_bits")
assert alice_indices_full == bob_indices_full, "alice and bob's matching indices don't match"

if GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists():
    # Already split in a previous run of this notebook -- reuse it rather
    # than re-splitting (re-splitting an already-split key would sample a
    # PE subset from what's actually generation-only bits, which is wrong).
    alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
    bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
    meta = _json.loads(META_PATH.read_text())
    k_pe, real_qber = meta["k"], meta["qber"]
    print(f"Reusing existing PE split: k={k_pe}, real_qber={real_qber:.4f}")
else:
    split = split_pe_and_generation(alice_key_full, bob_key_full, sample_fraction=0.1, seed=999001)
    alice_key, bob_key = split["alice_gen"], split["bob_gen"]
    k_pe, real_qber = split["k"], split["qber"]

    gen_indices = split["gen_indices"]
    gen_matching_indices = [alice_indices_full[idx] for idx in gen_indices]
    GEN_ALICE.write_text(_json.dumps({"alice_bits": alice_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    GEN_BOB.write_text(_json.dumps({"bob_bits": bob_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    META_PATH.write_text(_json.dumps({"k": k_pe, "qber": real_qber, "m": split["m"], "n": split["n"]}))

    print(f"m={split['m']} sifted, k={k_pe} PE sample, n={split['n']} generation bits, "
          f"real_qber (from disjoint PE sample) = {real_qber:.4f}")

# Re-sync both remote nodes to this exact (PE-split, generation-only) key pair.
alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes synced to PE-split generation-only key")

Reusing existing PE split: k=388, real_qber=0.0077
Remote nodes synced to PE-split generation-only key


## 1. Collect 5 real key pairs (FABRIC)

This replaces any standalone single-key PE split entirely -- key0 is now just `key_pairs_df.iloc[0]`, and every downstream cell iterates over all 5 rows, never a separate global `alice_key`/`k_pe`/`real_qber`.

In [ ]:
key_pairs_df = collect_key_pairs(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR, n_keys=5)
print(key_pairs_df)

## 2. Mock dose-response (n=50/point, per key), all four fault types

Loops over all 5 keys. Verify-digest split uses each key's own generation length and the **placeholder** length formula (not finite-key), computed per key since `n` differs by key.

In [6]:
import numpy as np

def h(Q):
    """Binary entropy, base 2."""
    if Q <= 0 or Q >= 1:
        return 0.0
    return -Q * np.log2(Q) - (1 - Q) * np.log2(1 - Q)

def asymptotic_key_length(n_bits, Q):
    """Asymptotic secret-key length: ell_inf/n -> 1 - 2h(Q),
    per Tomamichel & Leverrier (arXiv:1506.08458), Section 5."""
    return max(0, int(n_bits * (1 - 2 * h(Q))))

def placeholder_key_length(n_bits):
    """QFabric's default extraction-length formula (randextract library) --
    NOT from the finite-key security literature, a generic heuristic."""
    return ToeplitzHashing.calculate_length(
        extractor_type="quantum", input_length=n_bits,
        relative_source_entropy=0.5, error_bound=1e-6,
    )

def verification_t(eps_ec=1e-10):
    """t satisfying eps_ec = 2^-t (Theorem 2). Using t_verify = t directly
    (no multiplier), the simplification adopted earlier in this project."""
    return max(1, int(np.ceil(-np.log2(eps_ec))))

In [ ]:
mock_rows = []
for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    alice_key_i, _ = key_from_sifted_json(row["alice_path"], "alice_bits")
    bob_key_i, _ = key_from_sifted_json(row["bob_path"], "bob_bits")
    qber_i = float(row["qber"])
    n_bits_i = alice_key_i.get_nr_bits()

    t = verification_t()
    length_modes = {"asymptotic": asymptotic_key_length(n_bits_i, qber_i),
                     "placeholder": placeholder_key_length(n_bits_i)}

    for length_mode, ell in length_modes.items():
        for fault_type, probs, kwarg_name in [("toeplitz", probs_th, "toeplitz_prob"),
                                                ("final_key", probs_th, "final_key_prob"),
                                                ("reconciliation", probs_recon, "reconciliation_prob")]:
            for prob in probs:
                df_cond = run_condition_sweep(alice_key_i, bob_key_i, qber_i,
                                                 f"key{key_idx}_{length_mode}_{fault_type}_{prob}",
                                                 n_runs=20, ell=ell, t_verify=t, digest_length=t,
                                                 **{kwarg_name: prob})
                df_cond["fault_type"] = fault_type
                df_cond["prob"] = prob
                df_cond["length_mode"] = length_mode
                df_cond["key_index"] = key_idx
                df_cond["qber"] = qber_i
                mock_rows.append(df_cond)

mock_df = pd.concat(mock_rows, ignore_index=True)
mock_df.to_csv(str(PROJECT_DIR / "results" / "sdc_mock_allkeys_withverification.csv"), index=False)
print(f"Saved {len(mock_df)} rows")

In [ ]:
n_bits = alice_key.get_nr_bits()

t = verification_t()
t_verify = t
digest_length = t

length_modes = {
    "asymptotic": asymptotic_key_length(n_bits, real_qber),
    "placeholder": placeholder_key_length(n_bits),
}

all_dfs, summary_rows = {}, []

for length_mode, ell in length_modes.items():
    print(f"\n=== length_mode={length_mode}, ell={ell}, t_verify={t_verify} ===")
    for fault_type, probs, kwarg_name in [("toeplitz", probs_th, "toeplitz_prob"),
                                            ("final_key", probs_th, "final_key_prob"),
                                            ("reconciliation", probs_recon, "reconciliation_prob")]:
        for prob in probs:
            label = f"{length_mode}_{fault_type}_{prob}"
            print(f"  Running {label} (n=50)...")
            df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=50,
                                             ell=ell, t_verify=t_verify, digest_length=digest_length,
                                             **{kwarg_name: prob})
            df_cond["fault_type"] = fault_type
            df_cond["prob"] = prob
            df_cond["length_mode"] = length_mode
            all_dfs[label] = df_cond
            summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_mock_doseresponse_withverification.csv"), index=False)
print(f"\nSaved {len(full_df)} rows -> sdc_mock_doseresponse_withverification.csv")

In [9]:
n_bits = alice_key.get_nr_bits()
ell_asymp = asymptotic_key_length(n_bits, real_qber)
ell_placeholder = placeholder_key_length(n_bits)

print(f"n_bits = {n_bits}, Q = {real_qber}")
print(f"asymptotic ell = {ell_asymp}")
print(f"placeholder ell = {ell_placeholder}")
print(f"Equal? {ell_asymp == ell_placeholder}")

n_bits = 3488, Q = 0.007731958762886598
asymptotic ell = 3032
placeholder ell = 1706
Equal? False


In [10]:
from qne.cascade.finite_key import h, v
from scipy.stats import norm

class TrackedMockClassicalSession(MockClassicalSession):
    """Drop-in MockClassicalSession that also tallies leaked bits, without
    touching the library source. Remove this and use the library class
    directly once total_leaked_bits is added to qne.cascade permanently."""
    def __init__(self, correct_key):
        super().__init__(correct_key)
        self.total_leaked_bits = 0

    def ask_correct_parities(self, blocks):
        self.total_leaked_bits += len(blocks)
        super().ask_correct_parities(blocks)


def make_synthetic_pair(n_bits, qber, seed):
    alice_synth = Key(nr_bits=n_bits, seed=seed)
    bob_synth = alice_synth.copy()
    bob_synth.apply_noise(bit_error_rate=qber, seed=seed + 1)
    return alice_synth, bob_synth


def measure_epsilon(n_bits, qber, n_trials=200, base_seed=1000):
    failures = 0
    for run in range(n_trials):
        seed = base_seed + run
        a, b = make_synthetic_pair(n_bits, qber, seed)
        session = TrackedMockClassicalSession(correct_key=a)
        reconciliation = Reconciliation(
            algorithm=ORIGINAL, classical_session=session, noisy_key=b,
            estimated_bit_error_rate=qber, seed=seed + 100,
        )
        try:
            bob_reconciled = reconciliation.reconcile()
            if a.nr_bits_different(bob_reconciled) > 0:
                failures += 1
        except RuntimeError:
            failures += 1
    return failures / n_trials


def sweep_leakage(n_values, q_values, n_trials=30, base_seed=2000):
    rows = []
    for n_bits in n_values:
        for qber in q_values:
            leaked = []
            for run in range(n_trials):
                seed = base_seed + run
                a, b = make_synthetic_pair(n_bits, qber, seed)
                session = TrackedMockClassicalSession(correct_key=a)
                reconciliation = Reconciliation(
                    algorithm=ORIGINAL, classical_session=session, noisy_key=b,
                    estimated_bit_error_rate=qber, seed=seed + 100,
                )
                try:
                    bob_reconciled = reconciliation.reconcile()
                    leaked.append(session.total_leaked_bits)
                except RuntimeError:
                    continue
            if leaked:
                rows.append({"n": n_bits, "Q": qber,
                              "mean_leakage": np.mean(leaked), "n_trials_used": len(leaked)})
    return rows


def fit_xi1_xi2(rows, epsilon):
    z = norm.ppf(1 - epsilon)
    X, y = [], []
    for row in rows:
        n_bits, Q, leak = row["n"], row["Q"], row["mean_leakage"]
        X.append([n_bits * h(Q), np.sqrt(n_bits * v(Q)) * z])
        y.append(leak)
    X, y = np.array(X), np.array(y)
    (xi1, xi2), _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return xi1, xi2, X, y


n_values = [1000, 3000, 5000]
q_values = [0.005, 0.01, 0.02, 0.03, 0.05]

print("Measuring epsilon...")
epsilon = measure_epsilon(n_bits=3400, qber=0.015, n_trials=200)
print(f"epsilon = {epsilon:.4f}")
if epsilon == 0.0:
    print("WARNING: epsilon = 0 exactly -- Phi^-1(1) undefined. "
          "Increase n_trials, or use a conservative published fallback (e.g. 1e-2).")

print("\nSweeping leakage across (n, Q)...")
rows = sweep_leakage(n_values, q_values, n_trials=30)
for r in rows:
    print(f"  n={r['n']}, Q={r['Q']:.3f}: mean_leakage={r['mean_leakage']:.1f} "
          f"({r['n_trials_used']} valid trials)")

print("\nFitting xi1, xi2...")
xi1, xi2, X, y = fit_xi1_xi2(rows, epsilon)
print(f"xi1 = {xi1:.4f}")
print(f"xi2 = {xi2:.4f}")
print(f"Sanity check: published xi1 range is [1.05, 1.16] -- "
      f"{'plausible' if 1.0 <= xi1 <= 1.3 else 'CHECK THIS, seems off'}")

predicted = X @ np.array([xi1, xi2])
residuals = y - predicted
print(f"Mean absolute residual: {np.mean(np.abs(residuals)):.2f} bits")
print(f"Mean relative residual: {np.mean(np.abs(residuals) / y) * 100:.2f}%")

Measuring epsilon...
epsilon = 0.0050

Sweeping leakage across (n, Q)...
  n=1000, Q=0.005: mean_leakage=100.7 (30 valid trials)
  n=1000, Q=0.010: mean_leakage=188.8 (30 valid trials)
  n=1000, Q=0.020: mean_leakage=335.2 (30 valid trials)
  n=1000, Q=0.030: mean_leakage=457.1 (30 valid trials)
  n=1000, Q=0.050: mean_leakage=685.8 (30 valid trials)
  n=3000, Q=0.005: mean_leakage=315.9 (30 valid trials)
  n=3000, Q=0.010: mean_leakage=563.3 (30 valid trials)
  n=3000, Q=0.020: mean_leakage=989.4 (30 valid trials)
  n=3000, Q=0.030: mean_leakage=1372.8 (30 valid trials)
  n=3000, Q=0.050: mean_leakage=2039.2 (30 valid trials)
  n=5000, Q=0.005: mean_leakage=522.8 (30 valid trials)
  n=5000, Q=0.010: mean_leakage=930.1 (30 valid trials)
  n=5000, Q=0.020: mean_leakage=1644.1 (30 valid trials)
  n=5000, Q=0.030: mean_leakage=2289.9 (30 valid trials)
  n=5000, Q=0.050: mean_leakage=3399.9 (30 valid trials)

Fitting xi1, xi2...
xi1 = 2.3948
xi2 = -0.2088
Sanity check: published xi1 range 

In [ ]:
from qne.cascade.finite_key import h, v, finite_key_output_length, optimize_nu

def ell_finite_key_actual(n, k_pe, Q):
    ell, r, t, nu = finite_key_output_length(n, k_pe, Q)
    ell = int(max(0, round(ell)))
    t_verify = int(max(1, round(t)))  # also cast t/t_verify, same risk
    return ell, t_verify
    
mock_fk_rows = []
for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    alice_key_i, _ = key_from_sifted_json(row["alice_path"], "alice_bits")
    bob_key_i, _ = key_from_sifted_json(row["bob_path"], "bob_bits")
    qber_i = float(row["qber"])
    n_bits_i = alice_key_i.get_nr_bits()
    k_pe_i = int(row["k_pe"])

    ell, t_verify = ell_finite_key_actual(n_bits_i, k_pe_i, qber_i)
    digest_length = t_verify  # matches your t_verify=t simplification

    print(f"key{key_idx}: ell={ell}, t_verify={t_verify}")

    for fault_type, probs, kwarg_name in [("toeplitz", probs_th, "toeplitz_prob"),
                                            ("final_key", probs_th, "final_key_prob")]:
        for prob in probs:
            df_cond = run_condition_sweep(alice_key_i, bob_key_i, qber_i,
                                             f"key{key_idx}_finitekey_{fault_type}_{prob}",
                                             n_runs=50, ell=ell, t_verify=t_verify, digest_length=digest_length,
                                             **{kwarg_name: prob})
            df_cond["fault_type"] = fault_type
            df_cond["prob"] = prob
            df_cond["length_mode"] = "finite_key"
            df_cond["key_index"] = key_idx
            df_cond["qber"] = qber_i
            mock_fk_rows.append(df_cond)

mock_fk_df = pd.concat(mock_fk_rows, ignore_index=True)
mock_fk_df.to_csv(str(PROJECT_DIR / "results" / "sdc_mock_allkeys_finitekey.csv"), index=False)
print(f"Saved {len(mock_fk_df)} rows")

## 3. Real-channel sweeps, KEY-OUTER ordering

For each key, runs ALL FOUR fault types (both length modes) before moving to the next key. If interrupted after key0, you have a complete four-fault-type comparison for at least one key -- enough to validate the mechanistic story -- rather than one fault type spread thin across all 5 keys with no comparison at all.

In [ ]:
"""
Real-channel sweeps, all 4 fault types, KEY-OUTER ordering, ASYMPTOTIC MODE ONLY.

Placeholder mode dropped for all four fault types after confirming
statistical equivalence to asymptotic in prior collection (two-proportion
z-tests, all p > 0.5 for toeplitz/final_key; verify_digest is structurally
mode-independent by construction; reconciliation showed no evidence of
mode-dependence in the reconciliation_outcome comparison). This is a
deliberate scope decision, stated explicitly here and in the paper's
methods section -- not a silent omission.
"""
toeplitz_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv")
final_key_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_allkeys_asymptotic.csv")
recon_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_allkeys_asymptotic_fixed.csv")
verify_digest_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_allkeys_final.csv")

def load_completed(path, key_cols):
    if os.path.exists(path):
        df_existing = pd.read_csv(path)
        return set(zip(*[df_existing[c] for c in key_cols])), True
    return set(), False

def append_row(path, result, header_written):
    row_df = pd.DataFrame([result])
    row_df.to_csv(path, mode="a", header=not header_written, index=False)
    return True

toeplitz_completed, toeplitz_header = load_completed(toeplitz_output_path, ["key_index", "prob", "run"])
finalkey_completed, finalkey_header = load_completed(final_key_output_path, ["key_index", "prob", "run"])
recon_completed, recon_header = load_completed(recon_output_path, ["key_index", "prob", "run"])
verifydigest_completed, verifydigest_header = load_completed(verify_digest_output_path, ["key_index", "prob", "run"])

for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(row["k_pe"]), float(row["qber"])
    a_file, b_file = row["alice_path"], row["bob_path"]
    alice.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(b_file, "qfabric/results/bob_sifted_bits.json")
    print(f"\n{'=' * 20} KEY {key_idx} (qber={qber_i:.4f}) {'=' * 20}")

    print("  -- toeplitz (asymptotic only) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in toeplitz_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="asymptotic", toeplitz_prob=prob)
            result.update({"fault_type": "toeplitz", "prob": prob, "length_mode": "asymptotic",
                            "key_index": key_idx, "qber": qber_i})
            toeplitz_header = append_row(toeplitz_output_path, result, toeplitz_header)
            print(f"    prob={prob}, run={run}: keys_match={result.get('keys_match')} [saved]")

    print("  -- final_key (asymptotic only) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in finalkey_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="asymptotic", final_key_prob=prob)
            result.update({"fault_type": "final_key", "prob": prob, "length_mode": "asymptotic",
                            "key_index": key_idx, "qber": qber_i})
            finalkey_header = append_row(final_key_output_path, result, finalkey_header)
            print(f"    prob={prob}, run={run}: keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

    print("  -- reconciliation (asymptotic only, FIXED: real keys_match) --")
    for prob in probs_recon:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in recon_completed:
                continue
            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber_i, run, seed,
                                                               reconciliation_prob=prob, k=k_i,
                                                               length_mode="asymptotic")
            result.update({"fault_type": "reconciliation", "prob": prob, "length_mode": "asymptotic",
                            "key_index": key_idx, "qber": qber_i})
            recon_header = append_row(recon_output_path, result, recon_header)
            print(f"    prob={prob}, run={run}: non_convergent={result.get('non_convergent')}, "
                  f"keys_match={result.get('keys_match')} [saved]")

    print("  -- verify_digest (asymptotic only -- structurally mode-independent) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in verifydigest_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="asymptotic", verify_digest_prob=prob)
            result.update({"fault_type": "verify_digest", "prob": prob, "length_mode": "asymptotic",
                            "key_index": key_idx, "qber": qber_i})
            verifydigest_header = append_row(verify_digest_output_path, result, verifydigest_header)
            print(f"    prob={prob}, run={run}: keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

    print(f"\n  Key {key_idx} complete.")

print("\nAll keys, all fault types complete (asymptotic mode only).")
for path in [toeplitz_output_path, final_key_output_path, recon_output_path, verify_digest_output_path]:
    if os.path.exists(path):
        print(f"\n{path}:")
        print(pd.read_csv(path).groupby("key_index").size())

In [ ]:
toeplitz_fk_path = str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_allkeys_finitekey.csv")
final_key_fk_path = str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_allkeys_finitekey.csv")
recon_fk_path = str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_allkeys_finitekey.csv")
verify_digest_fk_path = str(PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_allkeys_finitekey.csv")

def load_completed(path, key_cols):
    if os.path.exists(path):
        df_existing = pd.read_csv(path)
        return set(zip(*[df_existing[c] for c in key_cols])), True
    return set(), False

def append_row(path, result, header_written):
    pd.DataFrame([result]).to_csv(path, mode="a", header=not header_written, index=False)
    return True

toeplitz_completed, toeplitz_header = load_completed(toeplitz_fk_path, ["key_index", "prob", "run"])
finalkey_completed, finalkey_header = load_completed(final_key_fk_path, ["key_index", "prob", "run"])
recon_completed, recon_header = load_completed(recon_fk_path, ["key_index", "prob", "run"])
verifydigest_completed, verifydigest_header = load_completed(verify_digest_fk_path, ["key_index", "prob", "run"])

for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(row["k_pe"]), float(row["qber"])
    a_file, b_file = row["alice_path"], row["bob_path"]
    alice.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(b_file, "qfabric/results/bob_sifted_bits.json")
    print(f"\n{'=' * 20} KEY {key_idx} (qber={qber_i:.4f}) {'=' * 20}")

    print("  -- toeplitz (finite_key) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in toeplitz_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="finite_key", toeplitz_prob=prob)
            result.update({"fault_type": "toeplitz", "prob": prob, "length_mode": "finite_key",
                            "key_index": key_idx, "qber": qber_i})
            toeplitz_header = append_row(toeplitz_fk_path, result, toeplitz_header)
            print(f"    prob={prob}, run={run}: non_convergent={result.get('non_convergent')}, "
                  f"keys_match={result.get('keys_match')} [saved]")

    print("  -- final_key (finite_key) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in finalkey_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="finite_key", final_key_prob=prob)
            result.update({"fault_type": "final_key", "prob": prob, "length_mode": "finite_key",
                            "key_index": key_idx, "qber": qber_i})
            finalkey_header = append_row(final_key_fk_path, result, finalkey_header)
            print(f"    prob={prob}, run={run}: non_convergent={result.get('non_convergent')}, "
                  f"keys_match={result.get('keys_match')} [saved]")

    print("  -- reconciliation (finite_key) --")
    for prob in probs_recon:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in recon_completed:
                continue
            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber_i, run, seed,
                                                               reconciliation_prob=prob, k=k_i,
                                                               length_mode="finite_key")
            result.update({"fault_type": "reconciliation", "prob": prob, "length_mode": "finite_key",
                            "key_index": key_idx, "qber": qber_i})
            recon_header = append_row(recon_fk_path, result, recon_header)
            print(f"    prob={prob}, run={run}: non_convergent={result.get('non_convergent')}, "
                  f"keys_match={result.get('keys_match')} [saved]")

    print("  -- verify_digest (finite_key) --")
    for prob in probs_th:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in verifydigest_completed:
                continue
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="finite_key", verify_digest_prob=prob)
            result.update({"fault_type": "verify_digest", "prob": prob, "length_mode": "finite_key",
                            "key_index": key_idx, "qber": qber_i})
            verifydigest_header = append_row(verify_digest_fk_path, result, verifydigest_header)
            print(f"    prob={prob}, run={run}: keys_match={result.get('keys_match')} [saved]")

    print(f"\n  Key {key_idx} complete.")

In [11]:
recon_path = str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_allkeys_asymptotic_fixed.csv")

REALCHANNEL_N_RUNS = 100  # up from ~10 -- resolves the coin-flip-level noise from tonight
probs_recon = [0.001, 0.003, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5]  # keep your full sweep, or
# probs_recon = [0.001, 0.003, 0.01, 0.03]  # restrict to just the low end if you want this to run faster

def load_completed(path, key_cols):
    if os.path.exists(path):
        df_existing = pd.read_csv(path)
        return set(zip(*[df_existing[c] for c in key_cols])), True
    return set(), False

def append_row(path, result, header_written):
    pd.DataFrame([result]).to_csv(path, mode="a", header=not header_written, index=False)
    return True

recon_completed, recon_header = load_completed(recon_path, ["key_index", "prob", "run"])

for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(row["k_pe"]), float(row["qber"])
    a_file, b_file = row["alice_path"], row["bob_path"]
    alice.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(a_file, "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(b_file, "qfabric/results/bob_sifted_bits.json")
    print(f"\n{'=' * 20} KEY {key_idx} (qber={qber_i:.4f}) {'=' * 20}")

    for prob in probs_recon:
        for run in range(REALCHANNEL_N_RUNS):
            if (key_idx, prob, run) in recon_completed:
                continue
            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber_i, run, seed,
                                                               reconciliation_prob=prob, k=k_i,
                                                               length_mode="asymptotic")
            result.update({"fault_type": "reconciliation", "prob": prob, "length_mode": "asymptotic",
                            "key_index": key_idx, "qber": qber_i})
            recon_header = append_row(recon_path, result, recon_header)
            print(f"    prob={prob}, run={run}: non_convergent={result.get('non_convergent')}, "
                  f"keys_match={result.get('keys_match')} [saved]")

    print(f"\n  Key {key_idx} complete.")


==================== KEY 0 (qber=0.0128) ====================
    prob=0.001, run=10: non_convergent=True, keys_match=None [saved]
    prob=0.001, run=11: non_convergent=True, keys_match=None [saved]
    prob=0.001, run=12: non_convergent=True, keys_match=None [saved]
    prob=0.001, run=13: non_convergent=True, keys_match=None [saved]


KeyboardInterrupt: 

In [18]:
stdout, _ = bob.execute("ps aux | grep bob_cascade_driver", quiet=True)
print(stdout)

ubuntu     37333  0.0  0.0   7764  3316 ?        Ss   13:54   0:00 bash -c ps aux | grep bob_cascade_driver
ubuntu     37335  0.0  0.0   7012  2236 ?        R    13:54   0:00 grep bob_cascade_driver



In [15]:
stdout, _ = bob.execute(
    "for f in ~/qfabric/results/bob_recon*.json; do "
    "grep -l '\"non_convergent\": true' \"$f\" 2>/dev/null; done | head -5",
    quiet=True
)
print(stdout)

In [12]:
# On one failing trial, check what actually got logged
stdout, _ = bob.execute("cat ~/qfabric/results/bob_<your_failing_file>.json", quiet=True)
data = json.loads(stdout)
print(data.get("non_convergent"), data.get("error"), data.get("elapsed_seconds"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [13]:
# Bare reconciliation, zero faults, using the SAME patched reconciliation.py
session = MockClassicalSession(correct_key=alice_key)
injector = SDCFaultInjector(reconciliation_state_prob=0.0, seed=1)  # no fault at all
recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                         estimated_bit_error_rate=real_qber, seed=1,
                         correct_key=alice_key, fault_injector=injector)
result = recon.reconcile()
print("converged cleanly:", alice_key.nr_bits_different(result) == 0)

converged cleanly: True


In [20]:
recon_df = pd.read_csv(recon_path)
bad_mask = recon_df["error"].str.contains("arbitrary_bit_prob", na=False)
print(f"{bad_mask.sum()} rows affected")
recon_df[~bad_mask].to_csv(recon_path, index=False)

AttributeError: Can only use .str accessor with string values, not floating

In [23]:
print(recon_df.loc[recon_df["seed"].astype(str).str.contains("arbitrary_bit_prob", na=False), "seed"])

400    Traceback (most recent call last):\n  File "/h...
401    Traceback (most recent call last):\n  File "/h...
402    Traceback (most recent call last):\n  File "/h...
403    Traceback (most recent call last):\n  File "/h...
Name: seed, dtype: str


In [24]:
print(recon_df.dtypes)
print(recon_df[recon_df["non_convergent"] == True][["key_index", "prob", "run", "elapsed_seconds", "seed"]])

toeplitz_prob                            float64
final_key_prob                           float64
reconciliation_prob                      float64
verify_digest_prob                           str
seed                                         str
output_path                                  str
k_pe                                     float64
total_corrections                            str
secure_key_length                            str
t_verify                                     str
digest_length                            float64
verification_passed                          str
remaining_errors_after_reconciliation      int64
non_convergent                               str
error                                    float64
elapsed_seconds                          float64
faults_fired                                 str
length_mode                                  str
keys_match                                object
alice_error                              float64
alice_output_path   

In [25]:
nonconv = recon_df[recon_df["non_convergent"] == True]
print(nonconv["elapsed_seconds"].describe())
print(nonconv.sort_values("elapsed_seconds").head(20)[["key_index", "prob", "run", "elapsed_seconds"]])

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: elapsed_seconds, dtype: float64
Empty DataFrame
Columns: [key_index, prob, run, elapsed_seconds]
Index: []


## 4. Completeness check (all keys)

In [ ]:
files_to_check = [
    "sdc_mock_doseresponse_allkeys.csv",
    "sdc_realchannel_toeplitz_allkeys.csv",
    "sdc_realchannel_finalkey_allkeys.csv",
    "sdc_realchannel_reconciliation_allkeys.csv",
    "sdc_realchannel_verifydigest_allkeys.csv",
    "key_pairs_metadata.csv",
]

for fname in files_to_check:
    path = PROJECT_DIR / "results" / fname
    if not path.exists():
        print(f"{fname}: NOT FOUND\n")
        continue

    df = pd.read_csv(str(path))
    print(f"=== {fname} ({len(df)} rows) ===")

    if "key_index" in df.columns and "length_mode" in df.columns:
        print(df.groupby(["key_index", "length_mode"]).size().unstack(fill_value=0))
    elif "key_index" in df.columns and "fault_type" in df.columns:
        print(df.groupby(["key_index", "fault_type"]).size().unstack(fill_value=0))
    elif "prob" in df.columns:
        print(df.groupby("prob").size())

    if "verification_passed" in df.columns:
        n_with_verification = df["verification_passed"].notna().sum()
        print(f"  Rows with verification data: {n_with_verification}/{len(df)}")
    print()


## 5. Plots -- pooled across keys, with QBER noted as a confound

**Caveat**: pooling across all 5 keys treats each key as a replicate of the same condition, but the 5 keys have genuinely different QBER and n. This plot pools them for an overall picture; a proper analysis should treat QBER as a covariate (e.g. in the mechanistic/GLM model from your plan) rather than ignoring the variation. Faceted per-key breakdown is included below the pooled plot.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats


def wilson_ci(successes, n, confidence=0.95):
    if n == 0:
        return 0.0, 0.0, 0.0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half_width = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - half_width), min(1, center + half_width)


def rate_by_prob(df, prob_col, success_col):
    probs = sorted(df[prob_col].unique())
    rates, lo, hi = [], [], []
    for p in probs:
        sub = df[df[prob_col] == p]
        n = len(sub)
        successes = sub[success_col].fillna(False).astype(bool).sum()
        rate, rlo, rhi = wilson_ci(successes, n)
        rates.append(rate); lo.append(rlo); hi.append(rhi)
    return probs, rates, lo, hi


mode_colors = {"placeholder": "tab:blue", "asymptotic": "tab:orange"}

df_toeplitz = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_allkeys.csv"))
df_finalkey = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_allkeys.csv"))
df_recon = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_allkeys.csv"))

df_toeplitz["mismatch"] = ~df_toeplitz["keys_match"].fillna(False)
df_finalkey["mismatch"] = ~df_finalkey["keys_match"].fillna(False)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for mode in ["placeholder", "asymptotic"]:
    sub = df_toeplitz[df_toeplitz["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[0].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[0].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[0].set_xscale('log'); axes[0].set_xlabel('Fault probability')
axes[0].set_ylabel('Mismatch rate (pooled across 5 keys)')
axes[0].set_title('Toeplitz-matrix fault')
axes[0].legend(); axes[0].grid(alpha=0.3)

for mode in ["placeholder", "asymptotic"]:
    sub = df_finalkey[df_finalkey["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[1].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[1].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[1].set_xscale('log'); axes[1].set_xlabel('Fault probability')
axes[1].set_title('Final-key fault')
axes[1].legend(); axes[1].grid(alpha=0.3)

for mode in ["placeholder", "asymptotic"]:
    sub = df_recon[df_recon["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "reconciliation_prob", "non_convergent")
    axes[2].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[2].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[2].set_xscale('log'); axes[2].set_xlabel('Reconciliation fault probability')
axes[2].set_ylabel('Non-convergence rate (pooled across 5 keys)')
axes[2].set_title('Reconciliation-state fault')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Real Channel, POOLED across 5 keys (n=%d/point/key/mode) -- QBER varies by key, not controlled for' % REALCHANNEL_N_RUNS)
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "results" / "fig_allkeys_pooled.png"), dpi=150)
plt.show()

# --- Per-key breakdown, so QBER variation is at least visible, not hidden ---
print("\n=== Per-key QBER and mismatch rate at toeplitz_prob=0.5, placeholder mode ===")
sub = df_toeplitz[(df_toeplitz["prob"] == 0.5) & (df_toeplitz["length_mode"] == "placeholder")]
per_key = sub.groupby("key_index").agg(qber=("qber", "first"), mismatch_rate=("mismatch", "mean"), n=("mismatch", "size"))
print(per_key.to_string())


In [25]:
for key_idx in range(len(key_pairs_df)):
    row = key_pairs_df.iloc[key_idx]
    n_bits_i = int(row["n_bits"])
    qber_i = float(row["qber"])
    ell_a = asymptotic_key_length(n_bits_i, qber_i)
    ell_p = placeholder_key_length(n_bits_i)
    print(f"key{key_idx}: n={n_bits_i}, Q={qber_i:.4f} -> ell_asymptotic={ell_a}, ell_placeholder={ell_p}, "
          f"{'SAME' if ell_a == ell_p else 'DIFFERENT'}")

key0: n=3504, Q=0.0128 -> ell_asymptotic=2810, ell_placeholder=1714, DIFFERENT
key1: n=3409, Q=0.0158 -> ell_asymptotic=2608, ell_placeholder=1666, DIFFERENT
key2: n=3360, Q=0.0027 -> ell_asymptotic=3180, ell_placeholder=1642, DIFFERENT
key3: n=3431, Q=0.0026 -> ell_asymptotic=3251, ell_placeholder=1677, DIFFERENT
key4: n=3456, Q=0.0052 -> ell_asymptotic=3131, ell_placeholder=1690, DIFFERENT


In [26]:
recon_mock = mock_df[mock_df["fault_type"] == "reconciliation"]
print(recon_mock[["prob", "length_mode", "secure_key_length", "t_verify"]].drop_duplicates())

       prob  length_mode  secure_key_length  t_verify
320   0.300   asymptotic               2810        34
340   0.100   asymptotic               2810        34
360   0.030   asymptotic               2810        34
380   0.010   asymptotic               2810        34
400   0.003   asymptotic               2810        34
420   0.001   asymptotic               2810        34
760   0.300  placeholder               1714        34
780   0.100  placeholder               1714        34
800   0.030  placeholder               1714        34
820   0.010  placeholder               1714        34
840   0.003  placeholder               1714        34
860   0.001  placeholder               1714        34
1200  0.300   asymptotic               2608        34
1220  0.100   asymptotic               2608        34
1240  0.030   asymptotic               2608        34
1260  0.010   asymptotic               2608        34
1280  0.003   asymptotic               2608        34
1300  0.001   asymptotic    

In [27]:
baseline_candidates = real_df[(real_df["prob"] <= 0.01) | (real_df["fault_type"] == "verify_digest")]
failing = baseline_candidates[baseline_candidates["mismatch"] == True]
print(failing.groupby("key_index").size())
print(failing[["key_index", "prob", "fault_type"]].drop_duplicates())

key_index
2    56
3    28
4    28
dtype: int64
      key_index   prob     fault_type
747           2  0.010       toeplitz
767           2  0.003       toeplitz
787           2  0.001       toeplitz
1078          3  0.010       toeplitz
1098          3  0.003       toeplitz
1118          3  0.001       toeplitz
1387          4  0.010       toeplitz
1407          4  0.003       toeplitz
1427          4  0.001       toeplitz
2347          2  0.010      final_key
2367          2  0.003      final_key
2387          2  0.001      final_key
2678          3  0.010      final_key
2698          3  0.003      final_key
2718          3  0.001      final_key
2987          4  0.010      final_key
3007          4  0.003      final_key
3027          4  0.001      final_key
5047          2  0.500  verify_digest
5067          2  0.300  verify_digest
5087          2  0.100  verify_digest
5107          2  0.050  verify_digest
5127          2  0.030  verify_digest
5147          2  0.010  verify_digest
516

In [29]:
baseline_check = real_df[(real_df["fault_type"].isin(["toeplitz", "final_key"])) &
                            (real_df["key_index"].isin([2, 3, 4])) &
                            (real_df["prob"].isin([0.001, 0.003, 0.01]))]

cols_to_check = ["key_index", "prob", "fault_type", "faults_fired",
                  "remaining_errors_after_reconciliation", "verification_passed", "keys_match"]
cols_present = [c for c in cols_to_check if c in baseline_check.columns]
print(f"Available: {cols_present}")
print(baseline_check[cols_present].drop_duplicates())

Available: ['key_index', 'prob', 'fault_type', 'faults_fired', 'remaining_errors_after_reconciliation', 'verification_passed', 'keys_match']
      key_index   prob fault_type faults_fired  \
740           2  0.010   toeplitz           {}   
747           2  0.010   toeplitz           {}   
751           2  0.010   toeplitz           {}   
760           2  0.003   toeplitz           {}   
767           2  0.003   toeplitz           {}   
771           2  0.003   toeplitz           {}   
780           2  0.001   toeplitz           {}   
787           2  0.001   toeplitz           {}   
791           2  0.001   toeplitz           {}   
1060          3  0.010   toeplitz           {}   
1078          3  0.010   toeplitz           {}   
1080          3  0.003   toeplitz           {}   
1098          3  0.003   toeplitz           {}   
1100          3  0.001   toeplitz           {}   
1118          3  0.001   toeplitz           {}   
1380          4  0.010   toeplitz           {}   
1387     